# 🌤️ Weather Assistant with openJiuwen (ReAct Agent)
本 notebook 展示如何基于 **openJiuwen** 框架，用不到 100 行代码构建一个会调用外部天气 API 的对话助手。
整体流程：
1. 定义工具（RESTful 天气查询）
2. 定义模型、提示词与 Agent 配置
3. 创建 ReActAgent
4. 单次查询 & 流式对话演示

## 0. 环境准备

In [ ]:
import os, sys
from datetime import datetime

# 如需把项目根目录加入 PYTHONPATH
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
os.environ.setdefault("LLM_SSL_VERIFY", "false")

# ===== 按需修改 =====
# API_BASE = os.getenv("API_BASE", "")
# API_KEY = os.getenv("API_KEY", "")
# MODEL_NAME = os.getenv("MODEL_NAME", "")
# MODEL_PROVIDER = os.getenv("MODEL_PROVIDER", "")

API_BASE = os.getenv("API_BASE", "your api base")
API_KEY = os.getenv("API_KEY", "your api key")
MODEL_NAME = os.getenv("MODEL_NAME", "your model name")
MODEL_PROVIDER = os.getenv("MODEL_PROVIDER", "your model provider")

WEATHER_URL = "your weather service url"  # 本地 mock 或真实地址

## 1. 定义工具（RESTful 天气查询）

In [ ]:
from openjiuwen.core.foundation.tool import RestfulApiCard, RestfulApi

def create_tool():
    weather_card = RestfulApiCard(
            id="weather_tool",
            name="WeatherReporter",
            description="天气查询插件",
            input_params={
                "type": "object",
                "properties": {
                    "location": {"description": "地点", "type": "string"},
                    "date": {"description": "日期", "type": "string"},
                },
                "required": ["location", "date"],
            },
            url=WEATHER_URL,
            headers={},
            method="GET",
        )
    weather_tool = RestfulApi(
        card=weather_card,
    )

    return weather_tool

## 2. 定义模型配置

In [ ]:
from openjiuwen.core.foundation.llm import ModelRequestConfig, ModelClientConfig

model_config = ModelRequestConfig(
    model=MODEL_NAME,
    temperature=0.6,
    top_p=0.8,
)
model_client = ModelClientConfig(
    client_provider=MODEL_PROVIDER,
    api_base=API_BASE,
    api_key=API_KEY,
    verify_ssl=False
)

## 3. 定义提示词

In [ ]:
def create_prompt_template():
    system_prompt = "你是一个AI助手，在适当的时候调用合适的工作流，帮助我完成任务！注意：只需要调用一次工作流后就进行总结，不要重复调用！"
    return [
        dict(role="system", content=system_prompt)
    ]

## 4. 创建 ReActAgent

In [ ]:
from openjiuwen.core.single_agent import AgentCard, ReActAgentConfig, ReActAgent
from openjiuwen.core.runner import Runner

agent_card = AgentCard(
        id="react_agent_1234",
        description="天气查询助手",
    )
prompt_template = create_prompt_template()
react_agent_config = ReActAgentConfig(
    model_client_config=model_client,
    model_config_obj=model_config,
    prompt_template=prompt_template
)
react_agent = ReActAgent(card=agent_card).configure(react_agent_config)
tool = create_tool()
Runner.resource_mgr.add_tool(tool)
react_agent.add_ability(tool.card)

## 5. 单次查询

In [ ]:
import asyncio

async def main():
    result = await Runner.run_agent(agent=react_agent,
                                    inputs={"query": "查询杭州天气", "conversation_id": "013"})
    print(result)

if __name__ == "__main__":
    asyncio.run(main())

## 6. 保存 / 加载 Agent（可选）

In [ ]:
# 保存配置到 json
with open("weather_agent.json", "w", encoding="utf-8") as f:
    f.write(react_agent_config.model_dump_json(indent=2))

# 下次直接读取
# from openjiuwen.agent.react_agent import load_react_agent_config
# agent_config = load_react_agent_config("weather_agent.json")

## 7. 小结 & 下一步
- ✅ 已演示 **工具定义 → Agent 创建 → 单次/流式调用** 完整闭环
- 🚀 下一步可尝试：  
  - 把天气服务换成真实 API（如 OpenWeather、和风）  
  - 增加记忆（MemoryStore）让 Agent 能追问  
  - 用 `@tool` 注解快速叠加更多本地函数 